# Econ 521 — Lab: Regression Discontinuity, Part 1
### The intuition, the picture, and a real sharp RD

**Friday, 2 October 2026 — 50 minutes**

Lecture this week covered the RD design (Mastering 'Metrics 4; Mixtape 6; The Effect 20).
Today we build the design from the ground up and then run one on real data.

**By the end of this lab you should be able to:**

1. Say, in a sentence about *people*, what has to be true for an RD to recover a causal effect.
2. Explain why "compare just above to just below" is a control strategy, and what it buys you
   that controlling for the running variable in a normal regression does not.
3. Recognize the single most common way a sharp RD produces a fake result — curvature in the
   running variable mistaken for a jump.
4. Estimate and plot a sharp RD on real data, and interpret the number you get, including what
   population it applies to.

**Plan**

| Minutes | Section |
|---|---|
| 0–3 | 0. Setup and where this fits |
| 3–15 | 1. Building the intuition with a simulation |
| 15–23 | 2. How a sharp RD lies to you |
| 23–43 | 3. Hansen (2015): punishment and deterrence in drunk driving |
| 43–50 | 4. Reporting an RD, and what's next week |

Cells marked **Your turn** have `# TODO` blanks. Work in pairs.

## 0. Setup and where this fits

Three weeks ago we had randomization do the work for us. Last week we used an instrument:
something that shifted treatment but had no other path to the outcome.

RD is the case where the instrument is *sitting in the assignment rule itself*. Somebody —
a legislature, an admissions office, a utility — wrote down a rule of the form

$$ W_i = \mathbf{1}\{X_i \ge c\} $$

and that rule is the only thing that changes discontinuously at $c$. $X$ is the **running
variable** (also: forcing variable, score), $c$ is the **cutoff**, $W$ is treatment.

The design is everywhere because bureaucracies love thresholds: test scores, poverty indices,
firm size, vote share, birth dates, blood alcohol content, pollution readings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

plt.rcParams.update({"figure.figsize": (7.5, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
rng = np.random.default_rng(521)
print("ready")

## 1. Building the intuition

Use the example from lecture. A Gifted and Talented program (GATE) admits students who score
**75 or above** on a test. We want the effect of GATE on adult earnings.

The problem is obvious: kids who score 75+ differ from kids who score 60 in ways that also
affect earnings. Call that `ability`. We don't observe it. In our simulation we *do*, because
we built it — that is the whole point of simulating first.

The truth below: **GATE raises earnings by exactly 10**.

In [ ]:
n, c = 2000, 75

test    = rng.uniform(55, 95, n)          # running variable X
ability = rng.normal(0, 5, n)             # the confounder we never see in real data
GATE    = (test >= c).astype(int)         # assignment rule: W = 1{X >= c}

earn = 20 + 0.5*test + 0.4*ability + 10*GATE + rng.normal(0, 6, n)
#           ^^^^^^^^   ^^^^^^^^^^^^   ^^^^^^^
#      smooth in X    confounding    TRUE EFFECT = 10

sim = pd.DataFrame({"test": test, "GATE": GATE, "earn": earn, "ability": ability})
sim["x"] = sim.test - c                   # recenter: x = 0 at the cutoff

naive = sim.loc[sim.GATE == 1, "earn"].mean() - sim.loc[sim.GATE == 0, "earn"].mean()
print(f"Naive treated - untreated : {naive:6.2f}")
print(f"Truth                     : {10:6.2f}")

The naive comparison is roughly double the truth. Not because of `ability` alone — mostly
because GATE students have higher test scores, and test scores pay. The confounder here is
the running variable itself.

So: look at the data.

In [ ]:
def binned_means(df, xcol, ycol, cutoff=0.0, nbins=20, window=None):
    d = df if window is None else df[df[xcol].abs() <= window]
    d = d.copy()
    lo, hi = d[xcol].min(), d[xcol].max()
    left  = pd.cut(d.loc[d[xcol] <  cutoff, xcol], np.linspace(lo, cutoff, nbins//2 + 1))
    right = pd.cut(d.loc[d[xcol] >= cutoff, xcol], np.linspace(cutoff, hi, nbins//2 + 1))
    d["bin"] = pd.concat([left, right])
    g = d.groupby("bin", observed=True).agg(x=(xcol, "mean"), y=(ycol, "mean"), n=(ycol, "size"))
    return g.dropna()

fig, ax = plt.subplots()
ax.scatter(sim.x, sim.earn, s=5, alpha=0.18, color="grey")
b = binned_means(sim, "x", "earn", nbins=20)
ax.scatter(b.x, b.y, s=45, color="black", zorder=3, label="binned means")
for side, col in [(sim.x < 0, "tab:blue"), (sim.x >= 0, "tab:red")]:
    s = sim[side]
    fit = np.polyfit(s.x, s.earn, 1)
    xs = np.linspace(s.x.min(), s.x.max(), 50)
    ax.plot(xs, np.polyval(fit, xs), color=col, lw=2.2)
ax.axvline(0, color="black", ls="--", lw=1)
ax.set_xlabel("test score - 75 (running variable)"); ax.set_ylabel("earnings")
ax.set_title("The RD picture: fit each side separately, read the gap at the cutoff")
ax.legend(); plt.show()

**This plot is the estimator.** Everything else is bookkeeping. Two local fits, one gap.

Note what we are *not* doing: we are not fitting one line through everything and adding a
dummy. We let the relationship between score and earnings differ on each side, because we
only care about the two fitted values at $x = 0$.

### 1.2 Narrowing the window

The formal argument is a limit: as you shrink toward the cutoff, the units on either side
become comparable in everything, not just in what you measured. Watch the estimate and the
standard error as the window shrinks.

In [ ]:
def rd_ols(df, y="earn", x="x", h=None, poly=1, cov="HC1"):
    d = df if h is None else df[df[x].abs() <= h]
    d = d.copy()
    d["D"] = (d[x] >= 0).astype(int)
    terms = ["D", x, f"D:{x}"]
    for p in range(2, poly + 1):
        terms += [f"I({x}**{p})", f"D:I({x}**{p})"]
    m = smf.ols(f"{y} ~ " + " + ".join(terms), data=d).fit(cov_type=cov)
    return pd.Series({"h": h, "N": int(m.nobs), "tau": m.params["D"], "se": m.bse["D"]})

bw = pd.DataFrame([rd_ols(sim, h=h) for h in [20, 15, 10, 5, 3, 2, 1]])
print(bw.assign(N=bw.N.astype(int)).round(3).to_string(index=False))

Two things happen at once and they pull against each other:

* **Bias falls.** Fewer and fewer units far from the cutoff are being used to pin down the
  fit at the cutoff, so a wrong functional form matters less.
* **Variance rises.** At $h = 1$ you have under a hundred observations.

That is the **bias–variance trade-off** that bandwidth selection exists to resolve. Next week
we let `rdrobust` pick $h$ by minimizing MSE instead of eyeballing it.

### 1.3 The assumption, in words

> **Continuity.** Everything *other than* the treatment that affects earnings would have moved
> smoothly through a score of 75. A student who scored 74 and a student who scored 76 are, on
> average, the same kind of student.

Said the other way: the only thing that may jump at 75 is the program. Formally
$\mathbb{E}[Y_i(0)\mid X=x]$ and $\mathbb{E}[Y_i(1)\mid X=x]$ are continuous at $c$, and then

$$ \tau_{SRD} = \lim_{x \downarrow c}\mathbb{E}[Y_i\mid X_i=x] - \lim_{x \uparrow c}\mathbb{E}[Y_i\mid X_i=x] $$

is the average treatment effect **for units with $X_i = c$** — not for the program as a whole.
Keep saying "at the cutoff" out loud until it is automatic. Cattaneo & Titiunik (2022) call
this the continuity framework; the alternative is the *local randomization* framework, which
assumes that inside a small window $[c-w, c+w]$ assignment is as good as a coin flip.

### Your turn (Exercise 1)

The design dies if units can *sort* across the cutoff. Simulate that: suppose students whose
raw score lands just below 75 can lobby to get bumped over, and the ones who lobby are the
high-`ability` ones.

Modify the DGP so that students with `ability > 3` and a raw score in `[72, 75)` get pushed to
75, then re-estimate. What happens to the RD estimate, and *why* does the continuity assumption
fail here?

In [ ]:
sim2 = sim.copy()

# TODO 1: build the manipulated running variable.
#   bump = (sim2.ability > 3) & (sim2.test >= 72) & (sim2.test < 75)
#   sim2["test_m"] = np.where(bump, 75.0, sim2.test)
#   sim2["x"] = ...                      # recenter on 75
#   sim2["GATE_m"] = ...                 # assignment under the manipulated score
#   sim2["earn_m"] = 20 + 0.5*sim2.test + 0.4*sim2.ability + 10*sim2.GATE_m + rng.normal(0,6,len(sim2))

# TODO 2: estimate the RD on (earn_m, x) in a window of 5 and compare with the truth of 10.

# TODO 3: plot a histogram of test_m with 1-point bins around 75.
#         What do you see at 75 that you did not see before?

## 2. How a sharp RD lies to you

Manipulation is the failure everyone warns you about. The failure that is actually more common
in published work is duller: **the relationship between $X$ and $Y$ is curved, you fit lines,
and the lines disagree at the cutoff.**

Below there is **no treatment effect at all**. The outcome is a smooth, curved function of the
running variable plus noise. Nothing happens at zero.

In [ ]:
m = 4000
xr = rng.uniform(-1, 1, m)
y_curved = 2*np.log(xr + 1.1) + rng.normal(0, 0.2, m)   # smooth, curved, NO jump
fake = pd.DataFrame({"x": xr, "y": y_curved, "D": (xr >= 0).astype(int)})

global_fit = smf.ols("y ~ D + x", data=fake).fit(cov_type="HC1")
print(f"Global linear, one slope : tau = {global_fit.params['D']:+.3f} "
      f"(se {global_fit.bse['D']:.3f})   <-- 'significant', and completely fake")
print(f"Truth                    : tau =  0.000")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
b = binned_means(fake, "x", "y", nbins=30)

axes[0].scatter(fake.x, fake.y, s=4, alpha=.12, color="grey")
axes[0].scatter(b.x, b.y, s=25, color="black")
fit = smf.ols("y ~ D + x", data=fake).fit()
for lo, hi, dd, col in [(-1, 0, 0, "tab:blue"), (0, 1, 1, "tab:red")]:
    xs = np.linspace(lo, hi, 50)
    axes[0].plot(xs, fit.params["Intercept"] + fit.params["D"]*dd + fit.params["x"]*xs, col, lw=2.2)
axes[0].set_title("Straight lines over the whole range\n-> a jump that is not there")

w = fake[fake.x.abs() <= 0.15]
axes[1].scatter(w.x, w.y, s=6, alpha=.3, color="grey")
loc = smf.ols("y ~ D + x + D:x", data=w).fit(cov_type="HC1")
for lo, hi, dd, col in [(-.15, 0, 0, "tab:blue"), (0, .15, 1, "tab:red")]:
    xs = np.linspace(lo, hi, 50)
    axes[1].plot(xs, loc.params["Intercept"] + loc.params["D"]*dd
                 + (loc.params["x"] + loc.params["D:x"]*dd)*xs, col, lw=2.2)
axes[1].set_xlim(-1, 1)
axes[1].set_title(f"Local linear, h = 0.15\ntau = {loc.params['D']:+.3f} (se {loc.bse['D']:.3f})")
for a in axes: a.axvline(0, color="k", ls="--", lw=1); a.set_xlabel("running variable")
plt.tight_layout(); plt.show()

The fix is not "add more polynomial terms." Gelman & Imbens (2019) is a whole paper arguing
that high-order global polynomials are a *bad* idea in RD: they put strange weights on
observations far from the cutoff, they behave badly at boundary points, and the estimate moves
around with the order you happened to pick.

The fix is **go local**: a low-order polynomial (usually linear), estimated only near the
cutoff, with a sensible bandwidth. That is what the modern toolkit does, and it is what we
automate next week.

### Your turn (Exercise 2)

Trace out the bias as a function of bandwidth for the fake data above.

In [ ]:
# TODO: for h in a grid from 1.0 down to 0.05, estimate the local linear RD on `fake`
#       and plot tau(h) with a 95% band. Mark the truth (zero) with a horizontal line.
#       Roughly where does the bias become negligible, and what has happened to the SE by then?

hs = np.array([1.0, 0.8, 0.6, 0.4, 0.3, 0.2, 0.15, 0.1, 0.05])
# res = pd.DataFrame([rd_ols(fake, y="y", h=h) for h in hs])
# ...

## 3. Hansen (2015): punishment and deterrence

> Benjamin Hansen, "Punishment and Deterrence: Evidence from Drunk Driving,"
> *American Economic Review* 105(4), 2015. (On your syllabus reading list.)

**The institution.** A driver pulled over in Washington State blows into a breathalyzer. The
reading, BAC, determines punishment by rule:

* BAC $\ge$ **0.08** → DUI: fines, license suspension, mandatory classes.
* BAC $\ge$ **0.15** → aggravated DUI: all of the above, heavier, plus an ignition interlock.

**The question.** Does harsher punishment deter future drunk driving? Outcome: whether the
driver is caught driving drunk again (`recidivism`).

**Why an RD is credible here.** A driver at 0.079 and a driver at 0.081 drank about the same
amount. The reading depends on body weight, what they ate, how long since the last drink, the
calibration of the device. Nobody can blow a precise 0.079 on purpose. The *punishment* jumps
at 0.08; drinking behavior, remorse, and everything else do not.

Note what the treatment actually is: not "being drunk" (that is continuous in BAC) but
"being punished more."

In [ ]:
URL = "https://raw.githubusercontent.com/scunning1975/mixtape/master/hansen_dwi.dta"
try:
    dwi = pd.read_stata("data/hansen_dwi.dta")          # local copy from Canvas, if present
except Exception:
    dwi = pd.read_stata(URL)                            # otherwise pull it

dwi = dwi[["bac1", "recidivism", "male", "white", "aged", "acc", "year"]].copy()
dwi["x"] = dwi.bac1 - 0.08                              # centered running variable
dwi["D"] = (dwi.bac1 >= 0.08).astype(int)               # DUI threshold crossed

print(dwi.shape)
print(dwi.describe().T[["mean", "std", "min", "max"]].round(3))

Variables: `bac1` blood alcohol content on the first test, `recidivism` caught again,
`male` / `white` / `aged` driver characteristics, `acc` an accident was involved.

Base rate of recidivism is about 11%, so keep that denominator in mind when we read the
coefficient.

### 3.1 Look before you estimate

In [ ]:
win = dwi[(dwi.bac1 >= 0.03) & (dwi.bac1 <= 0.13)]      # Hansen's main window

fig, ax = plt.subplots()
edges = np.arange(0.03, 0.1301, 0.002)
grp = win.groupby(pd.cut(win.bac1, edges), observed=True).agg(
        bac=("bac1", "mean"), y=("recidivism", "mean"), n=("recidivism", "size")).dropna()
ax.scatter(grp.bac, grp.y, s=38, color="black", zorder=3)

for side, col in [(win.bac1 < 0.08, "tab:blue"), (win.bac1 >= 0.08, "tab:red")]:
    s = win[side]
    f = np.polyfit(s.bac1, s.recidivism, 1)
    xs = np.linspace(s.bac1.min(), s.bac1.max(), 40)
    ax.plot(xs, np.polyval(f, xs), color=col, lw=2.3)

ax.axvline(0.08, color="black", ls="--")
ax.set_xlabel("BAC on first test"); ax.set_ylabel("P(caught driving drunk again)")
ax.set_title("Hansen (2015): recidivism around the 0.08 DUI threshold\n(0.002-wide bins)")
plt.show()

Read the picture before you read any regression:

1. The underlying relationship is **downward sloping** — heavier drinkers in this sample
   recidivate *less*. If you had just compared people above and below 0.08 without accounting
   for the slope, you would have gotten the wrong answer for the wrong reason.
2. There is a visible **drop at 0.08**, of a few tenths of a percentage point on a base of
   about 11 points.
3. The bins are reasonably tight, so we have a lot of data near the cutoff. That is unusual
   and it is why this paper works.

### 3.2 Estimate it four ways

The specification from lecture:

$$ Y_i = \beta_0 + \tau \, D_i + \beta_1 (X_i - c) + \beta_2 D_i (X_i - c) + \varepsilon_i $$

$\tau$ is the jump. The interaction lets the slope differ on each side — which it should, since
there is no reason for the BAC–recidivism gradient to be the same for punished and unpunished
drivers.

In [ ]:
specs = {
    "linear, h=0.05 (Hansen)" : rd_ols(dwi, y="recidivism", h=0.05, poly=1),
    "linear, h=0.03"          : rd_ols(dwi, y="recidivism", h=0.03, poly=1),
    "linear, h=0.02"          : rd_ols(dwi, y="recidivism", h=0.02, poly=1),
    "quadratic, h=0.05"       : rd_ols(dwi, y="recidivism", h=0.05, poly=2),
}
tab = pd.DataFrame(specs).T
tab["t"] = tab.tau / tab.se
tab["% of base rate"] = 100 * tab.tau / dwi.recidivism.mean()
print(tab.assign(N=tab.N.astype(int)).round(4).to_string())

### 3.3 What the number means

Take the main estimate, about **−0.02**.

* **Units.** The outcome is a 0/1 indicator, so this is a drop of 2 percentage points in the
  probability of being caught drunk driving again.
* **Scale.** On a base rate of 10.7%, that is a reduction of roughly **20%**. Big.
* **Population.** Drivers whose BAC came in *right around 0.08*. It says nothing about the
  deterrent effect of punishing someone who blew 0.25, and nothing about people who were never
  pulled over. The "local" in LATE is doing real work here.
* **Treatment.** The whole punishment bundle jumps at once — fine, suspension, classes. The RD
  cannot tell you which component deters. This is the "many things jump at the threshold"
  problem from lecture, and it limits what policy advice you can give.
* **Direction of policy interest.** The estimate is informative about *moving the threshold*
  slightly, which is exactly the policy question legislatures actually face.

Notice the quadratic specification gives a visibly smaller number than the linear one. Which do
you report? "The one I liked" is not an answer, and that is the problem data-driven bandwidth
and polynomial selection is designed to solve. Next week.

### Your turn (Exercise 3)

Washington has a **second** threshold at BAC = 0.15 (aggravated DUI). Estimate the RD there.

Keep three things in mind:
* Use a window that does not reach back past 0.08, or you will have two treatments in it.
* The second threshold is layered on top of the first — everyone near 0.15 is already a DUI,
  so you are estimating the effect of the *extra* punishment.
* Plot it first.

In [ ]:
# TODO 1: build a sample for the 0.15 cutoff
# agg = dwi[(dwi.bac1 >= 0.10) & (dwi.bac1 <= 0.20)].copy()
# agg["x"] = ...
# agg["D"] = ...

# TODO 2: binned-means plot around 0.15

# TODO 3: estimate the local linear RD. Is the effect bigger or smaller than at 0.08?
#         Give an economic reason for the sign and size you find.

## 4. Reporting an RD

What a referee expects to see, in order. We have done the first three today:

1. **The institutional rule.** What is the running variable, who set the cutoff, when, and what
   exactly changes when you cross it.
2. **The picture.** Binned means plus local fits. If you cannot see it, be honest that you
   cannot see it.
3. **The estimate**, with the bandwidth and polynomial order stated, not buried.
4. **Evidence against manipulation** — a density test on the running variable.
5. **Balance** — the same RD run on predetermined covariates should give zeros.
6. **Robustness** — other bandwidths, other polynomial orders, placebo cutoffs, a donut.

Items 4–6 are next week, along with fuzzy RD (imperfect compliance — which is just the IV
machinery from last week, with "above the cutoff" as the instrument) and the special
headaches of using **time** as the running variable, which is what Davis (2008) does in the
Mexico City slides.

**Before next Friday:** skim the Cattaneo & Titiunik (2022) *Annual Review* piece, sections 3.2
and 4. You do not need the asymptotics; you need the vocabulary — MSE-optimal bandwidth, robust
bias-corrected inference, density test, placebo cutoff, donut hole.